# 🧠 Lesson 02: LLM Fundamentals

Understanding how the 'brain' of your agent actually works — tokens, prompts, temperature, and prompting patterns.

In [ ]:
import os, tiktoken
from openai import AzureOpenAI
from dotenv import load_dotenv
from rich.console import Console
from rich.table import Table
load_dotenv()
client = AzureOpenAI(
    api_key=os.getenv('AZURE_OPENAI_API_KEY'),
    azure_endpoint=os.getenv('AZURE_OPENAI_ENDPOINT'),
    api_version=os.getenv('AZURE_OPENAI_API_VERSION')
)
DEPLOYMENT = os.getenv('AZURE_OPENAI_DEPLOYMENT', 'gpt-4o')
console = Console()
console.print('[green]✅ Ready![/green]')

## 🔢 Part 1: Tokens — What LLMs Actually See

In [ ]:
enc = tiktoken.encoding_for_model('gpt-4o')

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

# Compare token counts
samples = {
    'Simple sentence': 'Hello, I want to learn about arrays.',
    'Code snippet': 'def binary_search(arr, target):\n    left, right = 0, len(arr)-1',
    'JSON data': '{"name": "Avnish", "topic": "Binary Search", "difficulty": "medium"}',
    'Hindi text': 'मैं AI engineer बनना चाहता हूँ।',
}

table = Table(title='Token Count Comparison')
table.add_column('Type', style='cyan')
table.add_column('Text', style='white')
table.add_column('Tokens', style='green')

for name, text in samples.items():
    table.add_row(name, text[:50]+'...', str(count_tokens(text)))

console.print(table)

## 🌡️ Part 2: Temperature — Creativity vs Precision

In [ ]:
def ask(prompt: str, temperature: float) -> str:
    r = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=100
    )
    return r.choices[0].message.content

question = 'In one sentence, what is a binary search tree?'

for temp in [0.0, 0.5, 1.0]:
    answer = ask(question, temp)
    console.print(f'[yellow]Temperature {temp}:[/yellow] {answer}\n')

## 💬 Part 3: Prompting Patterns for Agents

In [ ]:
def chat(system: str, user: str) -> str:
    r = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ],
        temperature=0,
        max_tokens=300
    )
    return r.choices[0].message.content

# Pattern 1: Zero-shot
zero_shot = chat(
    system='You are a DSA tutor.',
    user='Explain time complexity of merge sort'
)
console.print('[cyan]Zero-shot:[/cyan]', zero_shot[:200])

# Pattern 2: Chain-of-Thought
cot = chat(
    system='You are a DSA tutor. Always think step by step before answering.',
    user='What is the time complexity of merge sort? Think step by step.'
)
console.print('\n[magenta]Chain-of-Thought:[/magenta]', cot[:300])

In [ ]:
# Pattern 3: Few-shot prompting
few_shot_system = """
You are a DSA tutor. Always explain topics in this format:
CONCEPT: [one line definition]
TIME: [time complexity]
USE WHEN: [when to use it]
EXAMPLE: [real code example]

Example 1:
CONCEPT: Binary Search finds a target in a sorted array
TIME: O(log n)
USE WHEN: Array is sorted, need fast lookup
EXAMPLE: mid = (left + right) // 2

Example 2:
CONCEPT: Stack is LIFO data structure
TIME: O(1) push/pop
USE WHEN: Need to reverse, undo, or track nested structures
EXAMPLE: stack.append(x); stack.pop()
"""

few_shot_answer = chat(system=few_shot_system, user='Explain Quick Sort')
console.print('\n[green]Few-shot result:[/green]\n', few_shot_answer)

## 🎯 Pattern 4: ReAct Prompting (used in agents!)

ReAct = **Re**asoning + **Act**ing — the agent thinks out loud before acting.

In [ ]:
react_system = """
You are a problem-solving agent. For every question:
THOUGHT: [reason about what you need to do]
ACTION: [what tool to use, or 'none' if you know the answer]
OBSERVATION: [what the action returned, or 'n/a']
ANSWER: [your final answer]
"""

react_answer = chat(
    system=react_system,
    user='How many elements does a binary tree with height 3 have at maximum?'
)
console.print('[bold blue]ReAct Response:[/bold blue]\n', react_answer)

## ✅ Key Takeaways
- Temperature 0 = best for agents (deterministic)
- Few-shot > zero-shot for structured output
- ReAct pattern = foundation of all modern agents
- Tokens are the currency — count them carefully!